# LLMs & Generative AI — Session 1: Foundations & Prompt Engineering
### ITI · Instructor: Ahmed Abdelsalam

You've already built models that **recognise** things — a CNN that reads digits, YOLO that finds objects. Today we meet models that **create** things: Large Language Models.

By the end you'll know what an LLM actually does under the hood (it's simpler than you think), and how to write prompts that get good results out of one.

**Today's map**
1. Generative vs. what you've done so far
2. How text becomes numbers — tokens
3. Embeddings — giving words meaning
4. **How an LLM generates text** (the core idea)
5. Temperature — why answers change
6. Prompt engineering basics

> **Setup:** made for **Google Colab**. A GPU runtime is faster but not required: *Runtime → Change runtime type → GPU*.

## Setup — run this first

We'll use small open models that download free, with no API key and no account.

In [ ]:
!pip install transformers -q

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

## 1. Generative AI — what's different?

Everything you've built so far was **discriminative**: it took an input and chose an answer from a fixed set.

| | What you built before | What we build today |
|---|---|---|
| CNN on MNIST | image → one of 10 digits | |
| YOLO | image → boxes from 80 classes | |
| **LLM** | | text → **new text**, invented word by word |

A discriminative model **picks**. A generative model **produces**. The output isn't chosen from a list — it's constructed one piece at a time, and the number of possible outputs is effectively infinite.

**Large Language Model (LLM)** = a very large neural network trained on enormous amounts of text, whose one job is to **predict the next piece of text**. That's genuinely it. Everything else — answering questions, writing code, summarising — emerges from doing that one thing extremely well.

## 2. How text becomes numbers — tokens

Neural networks only do maths on numbers. For images that was easy: a pixel is already a number. Text isn't.

So we **tokenize**: split text into pieces (*tokens*) and give each piece an ID number. A token is often a word, but common chunks and word-pieces get their own tokens too.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Computer vision is amazing!"
ids = tokenizer.encode(text)

print("text  :", text)
print("tokens:", [tokenizer.decode([i]) for i in ids])
print("ids   :", ids)
print("count :", len(ids), "tokens")

Notice the tokens include the leading spaces — `" vision"` is one token. Now watch what happens with an unusual word:

In [ ]:
for word in ["cat", "hippopotamus", "Abdelsalam", "تعلم"]:
    ids = tokenizer.encode(word)
    print(f"{word:15s} -> {[tokenizer.decode([i]) for i in ids]}")

> Common words are a single token. Rare words get **split into pieces**. This is why:
> - LLMs sometimes misspell unusual names
> - they're bad at counting letters (they see chunks, not letters)
> - non-English text often costs more tokens — and API pricing is per token

**Context window** = the maximum number of tokens a model can hold at once. Everything — your prompt and its answer — must fit.

## 3. Embeddings — giving tokens meaning

A token ID like `5761` is just a label; the number itself means nothing. So each token is mapped to a list of numbers called an **embedding** — a vector that captures meaning.

The useful property: **words with similar meanings get similar vectors.** Let's verify that on the real model.

In [ ]:
from transformers import AutoModel

embed_model = AutoModel.from_pretrained("gpt2")
E = embed_model.get_input_embeddings().weight   # the embedding table

def vec(word):
    ids = tokenizer.encode(" " + word)
    return E[ids].mean(0)

def similarity(a, b):
    return torch.cosine_similarity(vec(a), vec(b), dim=0).item()

for a, b in [("king", "queen"), ("cat", "dog"), ("king", "computer"), ("cat", "computer")]:
    print(f"{a:9s} vs {b:9s}  similarity = {similarity(a, b):.3f}")

> Related words score high, unrelated words score low — and nobody programmed that. The model learned it from reading text. This is the same idea as the *feature maps* in your CNN: the network invents its own useful representation.

## 4. How an LLM generates text — the core idea

Here's the whole mechanism:

1. Read the text so far.
2. Predict a **probability for every possible next token**.
3. Pick one.
4. Add it to the text and go back to step 1.

That loop, repeated, writes essays. Let's actually look at step 2 — the model's real predictions.

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    logits = model(**inputs).logits[0, -1]     # scores for the NEXT token

probs = torch.softmax(logits, dim=-1)          # turn scores into probabilities
top = torch.topk(probs, 8)

print(f"Prompt: {prompt!r}\n")
print("The model's top 8 guesses for the next token:")
for p, i in zip(top.values, top.indices):
    bar = "█" * int(p.item() * 100)
    print(f"  {tokenizer.decode([i]):>12s}  {p.item()*100:5.1f}%  {bar}")

> Two lessons in one output:
> 1. This is **all** an LLM does — a probability distribution over the next token. No lookup, no database, no reasoning engine.
> 2. GPT-2 is small and old, so `Paris` isn't even its top guess. **Bigger models predict better.** That's most of what "GPT-4 is better than GPT-2" means.

Now let's run the loop and watch text appear, one token at a time.

In [ ]:
prompt = "Artificial intelligence is"
ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

print(prompt, end="")
for step in range(12):
    with torch.no_grad():
        logits = model(ids).logits[0, -1]
    next_id = logits.argmax()                       # always take the most likely
    print(tokenizer.decode([next_id]), end="")
    ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
print("\n\n(that was 12 loops of: predict -> pick -> append)")

## 5. Temperature — why the answer changes

In the loop above we always took the **most likely** token. Do that and the model is repetitive and boring — and gives the identical answer every time.

Instead we usually **sample** from the probabilities. **Temperature** controls how adventurous that sampling is:

| Temperature | Behaviour | Use for |
|---|---|---|
| ~0 | always the most likely token | facts, code, extraction |
| ~0.7 | balanced | general use |
| >1.2 | takes risks, more surprising | creative writing, brainstorming |

Let's see the same prompt at two temperatures.

In [ ]:
chat_id = "Qwen/Qwen2.5-0.5B-Instruct"          # small instruct model, ~1GB
chat_tok = AutoTokenizer.from_pretrained(chat_id)
chat_model = AutoModelForCausalLM.from_pretrained(chat_id).to(device)
print("Chat model ready.")

def ask(prompt, max_new_tokens=40, temperature=None):
    """Send a prompt to the chat model and return its reply."""
    messages = [{"role": "user", "content": prompt}]
    text = chat_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = chat_tok(text, return_tensors="pt").to(device)
    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=chat_tok.eos_token_id)
    if temperature is None:
        kwargs["do_sample"] = False                  # greedy: same answer every time
    else:
        kwargs.update(do_sample=True, temperature=temperature, top_p=0.95)
    with torch.no_grad():
        out = chat_model.generate(**inputs, **kwargs)
    reply = chat_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return reply.strip()

print(ask("Say hello in one short sentence."))

In [ ]:
prompt = "Write one opening line for a story about the sea."

for T in [0.2, 1.4]:
    print(f"--- temperature {T} ---")
    for run in range(2):
        print("  ", ask(prompt, max_new_tokens=28, temperature=T))
    print()

> Low temperature repeats itself and plays safe. High temperature varies wildly. **Neither is "correct"** — pick to match the task.

## 6. Prompt engineering basics

A **prompt** is your input. Because the model is only continuing your text, *how you write it changes what you get*. Prompt engineering is just writing instructions clearly enough that a literal-minded system can follow them.

Five techniques cover most of it:

**1. Be specific.** Say exactly what you want.
- ❌ "Tell me about Python"
- ✅ "In 3 bullet points, explain why Python is popular for machine learning, for a beginner"

**2. Give context.** Who is it for, what's the situation?
- ✅ "I'm a CS student preparing for a job interview. Explain overfitting in simple terms."

**3. Assign a role.** Sets the tone and depth.
- ✅ "You are a patient programming tutor. Explain what a for-loop does."

**4. Show examples (few-shot).** The most reliable trick — demonstrate the pattern instead of describing it.
```
Classify the sentiment.
Text: "I love this" -> positive
Text: "This is terrible" -> negative
Text: "It works fine" ->
```

**5. Specify the output format.** Ask for JSON, a table, exactly 3 bullets — and you'll usually get it.

### Zero-shot vs few-shot
- **Zero-shot** — just ask. Fast, fine for easy tasks.
- **Few-shot** — include 2–5 worked examples first. Much better for a specific format or an unusual task.

Let's compare a vague prompt against an engineered one.

In [ ]:
vague = "Tell me about transfer learning."

engineered = """You are teaching a student who has just trained their first CNN.
Explain transfer learning in exactly 3 short bullet points.
Use simple language and no mathematics."""

for name, p in [("VAGUE", vague), ("ENGINEERED", engineered)]:
    print(f"===== {name} =====")
    print(ask(p, max_new_tokens=90))
    print()

> The second prompt gives the model a role, an audience, a length, a format, and a constraint. That's five pieces of guidance instead of zero — and the answer is far more usable.

**Note on model size:** our 0.5B model is small, so the improvement is real but modest. In the lab you'll repeat this in ChatGPT or Claude, where the difference is dramatic.

## ✅ Recap

- **Generative** models produce new content; the CNN and YOLO you built only chose from a fixed set.
- Text becomes **tokens** (with IDs); rare words split into pieces, which explains several LLM quirks.
- **Embeddings** turn tokens into vectors where similar meanings sit close together.
- An LLM **predicts the next token, then repeats** — that single loop is the whole mechanism.
- **Temperature** trades reliability for creativity.
- **Prompt engineering** = be specific, give context, assign a role, show examples, fix the format.

**Next (Lab):** explore tokenization, inspect real next-token probabilities, run temperature experiments, and do a prompt-engineering worksheet comparing weak vs strong prompts. Open `LLM_Session1_Lab.ipynb`.

**Session 2 preview:** where LLMs go wrong — hallucination, limitations, responsible use — and how to chain prompts into a working AI workflow.